# MLSys Task 5：集群、成本与整合——大规模系统怎么建？

对应打卡 issue：[datawhalechina/llm-algo-leetcode #137](https://github.com/datawhalechina/llm-algo-leetcode/issues/137)

理论材料：

- [Scaling to 1000 GPUs](https://mlsysbook.ai/mlsysim/tutorials/06_scaling_1000_gpus.html)（3D 并行 / 通信 / 气泡 / 可靠性）
- [The 9M Question](https://mlsysbook.ai/mlsysim/tutorials/08_the_9m_question.html)（TCO / 电费 / 碳排放）
- [Declarative DSE](https://mlsysbook.ai/mlsysim/tutorials/12_design_space_exploration.html)（复习+进阶）

新工具：

- `DistributedModel` —— 3D 并行（DP/TP/PP）、通信与气泡
- `EconomicsModel` —— Capex/Opex/TCO
- `SustainabilityModel` —— 能耗、碳、水
- `ReliabilityModel` / `CheckpointModel` —— MTBF 与 checkpoint 策略
- YAML 集群评估 —— 3-lens scorecard（Feasibility + Performance + Macro）

全部为解析仿真，Colab CPU runtime 可跑。


In [ ]:
# Colab 每次 new runtime 需要重新安装（约 1 分钟，纯 CPU 即可，不需要 GPU）
%pip install -q "git+https://github.com/harvard-edge/cs249r_book.git@dev#subdirectory=mlsysim"


In [ ]:
import math
import warnings
warnings.filterwarnings("ignore")

import mlsysim
from mlsysim.core.units import Q_
from mlsysim.solvers import (DistributedModel, EconomicsModel, SustainabilityModel,
                             ReliabilityModel, CheckpointModel)
from mlsysim.show import table, info
from mlsysim.systems.types import Fleet

print("mlsysim 版本:", mlsysim.__version__)

llama70b = mlsysim.Models.Language.Llama3_70B
gpt4     = mlsysim.Models.Language.GPT4          # 1.76T total / ~280B active (MoE, 第三方估计)
DGX      = mlsysim.Systems.Nodes.DGX_H100        # 8 x H100, NVLink 900 GB/s
IB_NDR   = mlsysim.Systems.Fabrics.InfiniBand_NDR

dist = DistributedModel()
econ = EconomicsModel()
sust = SustainabilityModel()

def mk_fleet(count, name=None):
    return Fleet(name=name or f"{count} node(s)", node=DGX, count=count, fabric=IB_NDR)

fleet8  = mk_fleet(1, "1 node / 8 GPUs")
fleet32 = mk_fleet(4, "4 nodes / 32 GPUs")

# 训练态每参数字节数（混合精度 Adam）：fp16 权重 2 + 梯度 2 + FP32 主权重 4 + 动量 4 + 方差 4 = 16
BYTES_PER_PARAM_TRAIN = 16

def train_mem_gb_per_gpu(model, tp, pp):
    """一阶估计：状态按 TP*PP 分片，不含激活值（下界）。"""
    return model.parameters.to("count").magnitude * BYTES_PER_PARAM_TRAIN / (tp * pp) / 1e9


def mag(x, unit=None):
    """统一取数值：Quantity 可选转换单位；裸数字直接返回（兼容不同字段的类型差异）。"""
    if hasattr(x, "magnitude"):
        return x.to(unit).magnitude if unit else x.magnitude
    return x


W70_FP16_GB = llama70b.size_in_bytes().to("GB").magnitude


---
## E1 单节点 8×H100：3D 并行策略怎么选？

70B FP16 权重 140 GB，训练态（Adam）约 70.6B × 16B ≈ **1.1 TB**——单卡 80 GB 必然放不下，
必须靠 TP×PP 分片。下面穷举 8 卡上所有合法 (TP, PP)：先做**内存可行性筛选**，再看效率与吞吐。

**预测区**：哪个组合会是冠军？TP=8（通信走 NVLink）一定赢吗？____


In [ ]:
# E1: DistributedModel 在单节点上的策略扫描（推理态 vs 训练态两本账）
rows, records_t5 = [], []
for tp, pp in [(tp, pp) for tp in (1, 2, 4, 8) for pp in (1, 2, 4, 8)
               if tp * pp <= 8 and 8 % (tp * pp) == 0]:
    r = dist.solve(model=llama70b, fleet=fleet8, batch_size=64,
                   tp_size=tp, pp_size=pp, precision="fp16", efficiency=0.45,
                   seq_len=2048, overlap_comm=True)
    w_per_gpu = W70_FP16_GB / tp                          # 推理态：只有权重
    st_per_gpu = train_mem_gb_per_gpu(llama70b, tp, pp)   # 训练态：权重+梯度+Adam
    infer_ok, train_ok = w_per_gpu <= 80, st_per_gpu <= 80
    thr = mag(r.effective_throughput)
    records_t5.append({"tp": tp, "pp": pp, "thr": thr,
                       "infer_ok": infer_ok, "train_ok": train_ok})
    rows.append([tp, pp, r.parallelism.get("dp", "?"),
                 f"{w_per_gpu:.0f}", f"{st_per_gpu:.0f}",
                 "OK" if infer_ok else "OOM",
                 "OK" if train_ok else "OOM",
                 f"{r.scaling_efficiency:.0%}",
                 f"{thr:.2f}" if infer_ok else "-"])

table(["TP", "PP", "DP", "权重GB/卡", "权重+Adam GB/卡", "推理态", "训练态",
       "扩展效率", "吞吐(样本/s)"], rows)

ok = [x for x in records_t5 if x["infer_ok"]]
best8 = max(ok, key=lambda x: x["thr"])
n_train_ok = sum(1 for x in records_t5 if x["train_ok"])
print()
print(f"单节点推理态参考最优: TP={best8['tp']}, PP={best8['pp']} (吞吐 {best8['thr']:.2f} 样本/s)")
print(f"训练态可行组合数: {n_train_ok} —— 70B+Adam 参数状态 ~1.13TB，8 卡怎么切都放不下")


**E1 解读 —— 3D 并行的核心权衡（以及一个关键发现）**：

- **DP 降通信**：梯度 AllReduce 的消息量 ∝ 模型大小/TP；DP 不引入气泡，但受临界 batch size 限制
- **TP 降内存、走 NVLink**：权重/梯度/优化器都按 TP 分片，通信快；但看表：即使 TP=8，训练态每卡仍需 ~141 GB → **OOM**
- **“训练态全 OOM”是本题最重要的发现**：70B + Adam 的参数状态约 1.13 TB，单个 8 卡节点无论怎么切都装不下——所以真实的 70B 训练至少要跨节点分摊优化器状态（E2 的 32 卡），或用 ZeRO-Offload / 启用 checkpoint 等手段压缩；“推理放得下 ≠ 训练放得下”
- **PP 用层间串行换内存**，引入 (pp−1)/(m+pp−1) 的气泡；microbatch 越多气泡越小


---
## E2 扩展到多节点：通信开销如何增长

先在 **32 卡**上选出**训练态内存可行**的最优 (TP, PP)，再把节点数从 1 扫到 8，
观察 DP AllReduce 与气泡占比的变化（不满足整除性的规模会自动回退到能整除的策略并标注）。

**预测区**：通信占比会从 ____ % 涨到 ____ %


In [ ]:
# E2-a: 在 32 卡上选“训练态内存可行”的最优策略（tp*pp >= ~15 才装得下 Adam 状态）
records32 = []
for tp, pp in [(tp, pp) for tp in (1, 2, 4, 8) for pp in (1, 2, 4, 8)
               if tp * pp <= 32 and 32 % (tp * pp) == 0]:
    if train_mem_gb_per_gpu(llama70b, tp, pp) > 80:
        continue
    r = dist.solve(model=llama70b, fleet=fleet32, batch_size=max(64, 32),
                   tp_size=tp, pp_size=pp, precision="fp16", efficiency=0.45,
                   seq_len=2048, overlap_comm=True)
    records32.append({"tp": tp, "pp": pp, "thr": mag(r.effective_throughput)})

best32 = max(records32, key=lambda x: x["thr"])
tb, pb = best32["tp"], best32["pp"]
print(f"32 卡训练态可行组合 {len(records32)} 个；最优: TP={tb}, PP={pb} "
      f"(吞吐 {best32['thr']:.2f} 样本/s)")


# E2-b: 节点数扫描。整除性不满足时回退到能整除的组合，并在表中标注。
def choose_strategy(n_gpu):
    if n_gpu % (tb * pb) == 0:
        return tb, pb
    for fb_tp, fb_pp in [(4, 2), (2, 2), (2, 1), (1, 2), (1, 1)]:
        if n_gpu % (fb_tp * fb_pp) == 0:
            return fb_tp, fb_pp
    return 1, 1


def comm_frac(rr):
    return mag(rr.communication_latency, "ms") / mag(rr.step_latency_total, "ms")


rows, dist_by_count = [], {}
for n_nodes in [1, 2, 4, 8]:
    fl = mk_fleet(n_nodes)
    n_gpu = 8 * n_nodes
    tp_i, pp_i = choose_strategy(n_gpu)
    r = dist.solve(model=llama70b, fleet=fl, batch_size=max(64, n_gpu),
                   tp_size=tp_i, pp_size=pp_i, precision="fp16",
                   efficiency=0.45, seq_len=2048, overlap_comm=True)
    dist_by_count[n_nodes] = r
    strat = f"TP{tp_i}/PP{pp_i}"
    if (tp_i, pp_i) != (tb, pb):
        strat += " (回退)"
    rows.append([n_nodes, n_gpu, strat, r.parallelism.get("dp", "?"),
                 f"{mag(r.communication_latency, 'ms'):.1f} ms",
                 f"{comm_frac(r):.1%}",
                 f"{r.bubble_fraction:.1%}",
                 f"{r.scaling_efficiency:.1%}"])

table(["节点", "GPU", "采用策略", "DP", "通信耗时", "通信占比", "气泡占比", "扩展效率"], rows)

print()
print(f"通信占比 1 节点 -> 4 节点: {comm_frac(dist_by_count[1]):.1%} -> "
      f"{comm_frac(dist_by_count[4]):.1%}；扩展效率 "
      f"{dist_by_count[1].scaling_efficiency:.1%} -> {dist_by_count[4].scaling_efficiency:.1%}")


**E2 解读**：

- 单节点 DP=1 几乎没有 DP AllReduce；跨节点后 DP 变大，梯度要在 IB NDR（400Gb/s）上做**分层 AllReduce**（节点内 NVLink ring + 跨节点 ring），通信占比随之上涨
- 这是 **Amdahl 定律**的具象化：通信是串行分量，N 翻倍不能让步时间减半
- 注：1 节点行是“回退”策略（32 卡的最优组在 8 卡上放不下也除不尽），对比趋势时以同策略的 2→4→8 行为准
- 缓解手段（都是 `DistributedModel.solve` 的参数，可以自己再试）：`overlap_comm=True` 用计算盖住通信、`zero_stage` 分散优化器状态、增大 `microbatch_count` 压气泡


---
## E3 EconomicsModel：32×H100 训练 70B 一个月的 TCO

**预测区**：Capex 和 Opex 谁占大头？____


In [ ]:
# E3: 总拥有成本 = Capex(硬件采购摊销) + Opex(电费 + 运维)
tco = econ.solve(fleet=fleet32, duration_days=30, mfu=0.45)

total = tco.tco_usd
info("TCO 分析 (32 x H100, 30 天)",
     Capex=f"${tco.capex_usd:,.0f} ({tco.capex_usd/total:.0%})",
     Opex_电费=f"${tco.opex_energy_usd:,.0f} ({tco.opex_energy_usd/total:.0%})",
     Opex_运维=f"${tco.opex_maintenance_usd:,.0f} ({tco.opex_maintenance_usd/total:.0%})",
     总计=f"${total:,.0f}",
     总能耗=f"{tco.total_energy_kwh.to('MWh'):.0f} MWh",
     区域=tco.region_name)


**E3 解读 —— 为什么大规模训练的成本不是线性的（The 9M Question）**：

- 卡数翻倍 ≠ 成本翻倍：网络设备、整机柜、供电冷却（PUE）、运维人力都有自己的非线性阶梯
- **Capex 占大头**意味着真正的杠杆是**利用率**：同样的卡，MFU 从 0.3 提到 0.45 等效免费多出 50% 算力
- 电费看似小头，但换到脏电网或碳价内部化的地区就显著上浮——下一节把“碳”变成可见数字


---
## E4 SustainabilityModel：同一个训练，三地碳排放差几倍

**预测区**：魁北克(水电) / 美国均值 / 波兰(煤电)，碳足迹最大差距 ≈ ____ 倍？


In [ ]:
# E4: 地理位置是一等公民的系统变量
grids = [mlsysim.Infrastructure.Grids.Quebec,
         mlsysim.Infrastructure.Grids.Norway,
         mlsysim.Infrastructure.Grids.US_Avg,
         mlsysim.Infrastructure.Grids.Poland]

rows, carbon = [], {}
for g in grids:
    r = sust.solve(fleet=fleet32, duration_days=30, datacenter=g)
    carbon[r.region_name] = r.carbon_footprint_kg
    rows.append([r.region_name,
                 f"{r.total_energy_kwh.to('MWh'):.0f} MWh",
                 f"{r.carbon_footprint_kg/1000:.1f} t CO2",
                 f"{r.water_usage_liters/1000:.0f} kL 水",
                 f"PUE {r.pue:.2f}"])

table(["区域", "能耗", "碳排放", "水耗", "PUE"], rows)

vals = list(carbon.values())
print()
print(f"最脏/最干净之比: {max(vals)/min(vals):.0f}x")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(list(carbon.keys()), [v/1000 for v in carbon.values()], color="#16a34a")
ax.set_ylabel("tonnes CO2eq / 30 days")
ax.set_title("Same job, different grid")
ax.grid(axis="y", alpha=.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


**E4 解读**：能耗基本相同（差异只来自 PUE），**碳强度（gCO2/kWh）制造了几十倍的差距**。
选址是一个决策动作，效果胜过一切工程优化——这是大厂扎堆水电富集区建数据中心的原因。


---
## E5 YAML 集群评估：一键输出 3-lens scorecard

把集群写成声明式 YAML，`mlsysim eval` 一次性给出 **Feasibility（可行性）/ Performance（性能）/ Macro（经济+碳）** 三个镜头。


In [ ]:
# E5: 写 cluster.yaml
yaml_text = """
version: "1.0"
name: "Llama-3 70B on 32x H100 (Quebec, 30d)"

workload:
  name: "Llama3_70B"
  batch_size: 256
  seq_len: 2048

hardware:
  name: "H100"
  accelerators: 32
  precision: "fp16"
  efficiency: 0.45

ops:
  region: "Quebec"
  duration_days: 30.0
"""
with open("cluster.yaml", "w") as f:
    f.write(yaml_text)

print(yaml_text)


In [ ]:
# 3-lens scorecard（文本版）
!mlsysim eval cluster.yaml


In [ ]:
# JSON 版（便于程序化处理）
!mlsysim eval cluster.yaml -o json | head -50


**E5 解读 —— 为什么三个镜头要一起看**：

- Level 1 Feasibility：装不装得下（显存可行性）——不过关其他免谈
- Level 2 Performance：瓶颈在哪（Memory/Compute bound）、MFU 多少——技术好不好
- Level 3 Macro：TCO、电费、碳——**值不值、绿不绿**；YAML 里没写 ops 段时显示 SKIPPED（不是报错）
- 正确流程是把三层当成一张体检单：任何一层红灯，方案回炉


---
## O1（选做）ReliabilityModel + CheckpointModel：故障与存档的经济学

集群越大越易碎：单卡 MTTF ~5 万小时，N 卡集群 MTBF ≈ MTTF/N。
Young-Daly 公式给出最优 checkpoint 间隔 `T_opt = sqrt(2 × δ × M)`（δ=写一次 checkpoint 的时间，M=集群 MTBF）。

**预测区**：32 卡集群的 MTBF ≈ ____ 小时？30 天预期故障 ____ 次？


In [ ]:
# O1-a: 32 GPU / 30 天任务的可靠性画像
rel = ReliabilityModel().solve(fleet=fleet32, job_duration_hours=30*24, checkpoint_time_s=60.0)
mtbf_h = rel.fleet_mtbf.to("hour").magnitude
opt_h = rel.optimal_checkpoint_interval.to("hour").magnitude

info("可靠性分析 (32 x H100, 30 天)",
     集群MTBF=f"{mtbf_h:.1f} 小时",
     预期故障次数=f"{rel.expected_failures:.1f} 次",
     最优checkpoint间隔=f"{opt_h:.2f} 小时",
     Goodput=f"{rel.goodput_ratio:.1%}")

yd_h = math.sqrt(2 * 60 * mtbf_h * 3600) / 3600     # Young-Daly 校验
print()
print(f"Young-Daly 手算: sqrt(2 * 60s * {mtbf_h:.1f}h) = {yd_h:.2f} 小时（应与上面一致）")


In [ ]:
# O1-b: checkpoint 尺寸与 I/O 冲击（Adam: 每参数 14 字节）
ckpt = CheckpointModel()
rows, ckpt_by_interval = [], {}
for interval_h in [0.5, 1, 2, 4, 8]:
    c = ckpt.solve(model=llama70b, hardware=DGX.accelerator, optimizer="adam",
                   checkpoint_interval_hours=interval_h)
    ckpt_by_interval[interval_h] = c
    rows.append([f"{interval_h} h",
                 f"{c.checkpoint_size.to('GB'):.0f} GB",
                 f"{c.write_time_seconds.to('second'):.0f} s",
                 f"{c.mfu_penalty_pct:.2%}",
                 "是" if c.storage_bottleneck else "否"])

table(["checkpoint 间隔", "单份大小", "写入耗时", "MFU 损失", "存储成瓶颈?"], rows)

write_s_at_opt = ckpt_by_interval[min(ckpt_by_interval, key=lambda k: abs(k - opt_h))].write_time_seconds.to("second").magnitude
print()
print(f"按最优间隔 ~{opt_h:.2f}h 存档: 30 天约 {30*24/opt_h:.0f} 次 x {write_s_at_opt:.0f}s 写入，"
      f"外加每次故障平均回滚半个间隔的工作量")


**O1 解读**：checkpoint 不是免费的保险——写盘打断训练（MFU 损失列），太频繁浪费 I/O，太稀疏则故障回滚损失惨重。
Young-Daly 平衡两者；集群越大 MTBF 越短，最优间隔自动收紧。


---
## O2（选做）多目标 DSE：天数 × TCO × 碳的 Pareto 前沿

思路：固定训练总量（70B × 2T tokens），对 {集群规模 × 并行策略} 的每个可行配置计算：
① 训练天数（Iron Law：`C = 6PD`，可达算力 = N × 峰值 × MFU × 扩展效率η）；
② 该时长下的 TCO；③ 碳排放（美国均值电网）。
然后手动求三维 Pareto 前沿。

> 这里不用 DSE 引擎而用手动循环，是因为我们要保留**每个候选的三个指标**做支配关系判断（DSE 只回 Top-5）。


In [ ]:
# O2: 构造候选并求 Pareto 前沿
PEAK = 989e12                                       # H100 FP16 dense FLOP/s
P = llama70b.parameters.to("count").magnitude
C_total = 6 * P * 2e12                              # Chinchilla: C = 6PD
MFU = 0.45

candidates = []
for n_nodes in [1, 2, 4, 8, 16]:
    fl = mk_fleet(n_nodes)
    n_gpu = 8 * n_nodes
    for tp, pp in [(tp, pp) for tp in (1, 2, 4, 8) for pp in (1, 2, 4, 8)
                   if tp * pp <= n_gpu and n_gpu % (tp * pp) == 0]:
        if train_mem_gb_per_gpu(llama70b, tp, pp) > 80:
            continue                                 # 内存可行性筛选
        try:
            r = dist.solve(model=llama70b, fleet=fl, batch_size=max(64, n_gpu),
                           tp_size=tp, pp_size=pp, precision="fp16",
                           efficiency=MFU, seq_len=2048, overlap_comm=True)
        except Exception:
            continue
        days = C_total / (n_gpu * PEAK * MFU * r.scaling_efficiency) / 86400
        t = econ.solve(fleet=fl, duration_days=days, mfu=MFU)
        s = sust.solve(fleet=fl, duration_days=days,
                       datacenter=mlsysim.Infrastructure.Grids.US_Avg)
        candidates.append({"nodes": n_nodes, "tp": tp, "pp": pp,
                           "eta": r.scaling_efficiency, "days": days,
                           "tco": t.tco_usd, "carbon": s.carbon_footprint_kg})

print(f"共 {len(candidates)} 个可行候选")


def dominates(a, b):   # a 支配 b：三目标都不差且至少一项严格更好
    no_worse = (a["days"] <= b["days"] and a["tco"] <= b["tco"] and a["carbon"] <= b["carbon"])
    strictly = (a["days"] < b["days"] or a["tco"] < b["tco"] or a["carbon"] < b["carbon"])
    return no_worse and strictly


frontier = [c for c in candidates
            if not any(dominates(o, c) for o in candidates if o is not c)]
frontier.sort(key=lambda c: c["days"])

rows = [[c["nodes"], c["tp"], c["pp"], f"{c['eta']:.0%}", f"{c['days']:.1f}",
         f"${c['tco']/1e6:.2f}M", f"{c['carbon']/1000:.0f} t"] for c in frontier]
table(["节点", "TP", "PP", "eta", "训练天数", "TCO", "碳(t)"], rows)


In [ ]:
# O2: 可视化 + 平衡点选择
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.4))
xs = [c["days"] for c in candidates]
ys = [c["tco"]/1e6 for c in candidates]
cs = [c["carbon"]/1000 for c in candidates]
sc = ax.scatter(xs, ys, c=cs, cmap="viridis", s=42)
fig.colorbar(sc, label="carbon (tonnes)")
fx = [c["days"] for c in frontier]
fy = [c["tco"]/1e6 for c in frontier]
ax.plot(fx, fy, "r.--", lw=1, label="Pareto frontier")
ax.set_xlabel("training days (lower = better)")
ax.set_ylabel("TCO ($M, lower = better)")
ax.set_title("color = carbon (tonnes, lower = better)")
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()


# “平衡方案”：三指标归一化名次之和最小（不属于任何单目标最优，却最难被拒绝）
def ranks(vals):
    order = sorted(range(len(vals)), key=lambda i: vals[i])
    pos = [0] * len(vals)
    for rk, idx in enumerate(order):
        pos[idx] = rk
    return pos

rk_days = ranks([c["days"] for c in candidates])
rk_tco  = ranks([c["tco"] for c in candidates])
rk_carb = ranks([c["carbon"] for c in candidates])
balanced = min(candidates, key=lambda c: rk_days[candidates.index(c)] +
                                        rk_tco[candidates.index(c)] +
                                        rk_carb[candidates.index(c)])
print("平衡方案:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in balanced.items()})


**O2 结论框架**：

- 前沿上的点各有性格：最快（最贵）、最低 TCO（往往是中等规模）、最低碳（清洁电网 + 适度时长）
- 平衡方案的辩护词：“它在三个维度上都排进前 __%，任何单一目标的方案都会在其他维度付出 __ 倍代价”——工程决策区别于刷榜的地方就在这里


---
## O3（选做）GPT-4 级别（1.76T）在万卡集群上的时间-成本-碳预测

Iron Law：`Time = 6PD / (N × Peak × MFU × η)`。取 P=1.76T、D=1.5T tokens（issue 设定）、N=10000×H100。
η 取自 Production_2K（2048 卡）上 TP=8/PP=8 可行配置的扩展效率——**外推假设，报告里要注明**。

**预测区**：预计训练时长 ____ 天？TCO ____ 亿美元量级？


In [ ]:
# O3: 万卡推演
PEAK = 989e12
prod2k = mlsysim.Systems.Clusters.Production_2K      # 2048 GPUs
r_eta = dist.solve(model=gpt4, fleet=prod2k, batch_size=2048,
                   tp_size=8, pp_size=8, precision="fp16", efficiency=0.45,
                   seq_len=2048, overlap_comm=True)
eta = r_eta.scaling_efficiency

P_gpt4 = gpt4.parameters.to("count").magnitude       # 1.76e12
D_gpt4 = 1.5e12                                      # tokens (issue 设定)
C_gpt4 = 6 * P_gpt4 * D_gpt4

fleet10k = mlsysim.Systems.Clusters.Training_10K     # 10000 x H100
N = fleet10k.total_accelerators
days = C_gpt4 / (N * PEAK * 0.45 * eta) / 86400

t10k = econ.solve(fleet=fleet10k, duration_days=days, mfu=0.45)
s_qc = sust.solve(fleet=fleet10k, duration_days=days,
                  datacenter=mlsysim.Infrastructure.Grids.Quebec)
s_pl = sust.solve(fleet=fleet10k, duration_days=days,
                  datacenter=mlsysim.Infrastructure.Grids.Poland)

info("GPT-4 级别训练推演 (10000 x H100)",
     总计算量=f"{C_gpt4:.2e} FLOP",
     扩展效率=f"{eta:.1%} (测自 2K 卡 TP8/PP8, 外推假设)",
     预计时长=f"{days:.0f} 天",
     TCO=f"${t10k.tco_usd/1e6:,.0f}M (capex 占 {t10k.capex_usd/t10k.tco_usd:.0%})",
     能耗=f"{t10k.total_energy_kwh.to('GWh'):.1f} GWh",
     碳_Quebec=f"{s_qc.carbon_footprint_kg/1000:,.0f} t",
     碳_Poland=f"{s_pl.carbon_footprint_kg/1000:,.0f} t")


**O3 讨论**：

- 对照公开估计（GPT-4 ≈ 2500 万 A100-day）：我们用 1 万张 H100、η=__%、MFU 45% 得到 ____ 天——量级是否合理？差异来自哪些假设（tokens 数、MoE 只按总参数计 6PD、η 外推）？
- 成本结构：Capex 占 ____%；如果集群利用率再降 10 个百分点，账单如何变化？
- 选址：同一训练 Quebec vs Poland 相差 ____ 倍碳——CTO 的选址理由应该写在方案第一页


---
## 打卡对照清单（issue #137）

**最小打卡（E1–E3）**
- [ ] E1：单节点最优 DP/TP/PP 组合与吞吐 + 内存筛选的解释
- [ ] E2：1→4 节点通信占比从 __% 涨到 __%
- [ ] E3：30 天 TCO、Capex/Opex 比例、“成本非线性”一段话

**学有余力 1（+E4 E5 O1）**
- [ ] E4：三地碳排对比 + “选址单枪匹马改变碳足迹”
- [ ] E5：YAML 3-lens scorecard 输出 + 三镜头价值
- [ ] O1：MTBF、最优 checkpoint 间隔、MFU 冲击

**学有余力 2（+O2 O3）**
- [ ] O2：Pareto 前沿图 + 平衡方案辩护
- [ ] O3：GPT-4 级训练的时间-成本-碳完整预测（设计方案可用 O3 输出作骨架）

> 截图建议：scorecard 文本、碳排柱状图、Pareto 散点图是最有说服力的三张图。
